In [ ]:
import sys, os, glob, random, itertools, json
import numpy as np
import torch
import plotly.graph_objects as go

sys.path.insert(0, r"c:\repos\DroneDetectionRF")

from NoisyUAV.funciones.detector_entropia import detectar_bursts, plot_muestra
from NoisyUAV.modelos.burst_cvcnn import BurstCVCNN

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


In [ ]:
TEACHER_CKPT = r"c:\repos\DroneDetectionRF\NoisyUAV\curriculum_teacher_v1\checkpoints\teacher_model_best.pt"

ckpt = torch.load(TEACHER_CKPT, map_location=DEVICE, weights_only=False)
model = BurstCVCNN(dropout_cnn=0.3, dropout_fuse=0.4).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

phys_mean = torch.from_numpy(ckpt['phys_mean']).to(DEVICE)
phys_std  = torch.from_numpy(ckpt['phys_std']).to(DEVICE)

print(f"✅ Teacher cargado — Val F1 histórico: {ckpt.get('val_f1', '?'):.4f}")

In [ ]:
DURATION_REF_PATH = r"c:\repos\DroneDetectionRF\NoisyUAV\curriculum_alumn_v1\drone_duration_ref.json"

DURATION_REF = {}
if os.path.exists(DURATION_REF_PATH):
    with open(DURATION_REF_PATH) as f:
        DURATION_REF = json.load(f)
    print(f"✅ Referencia de duraciones: targets {list(DURATION_REF.keys())}")
else:
    print("⚠️  drone_duration_ref.json no encontrado. El oráculo correrá sin restricción de duración mínima.")
    print("   Ejecuta: python NoisyUAV/curriculum_alumn_v1/calibrate_drone_durations.py")


In [ ]:
DURATION_REF["6"]

In [ ]:
def cargar_muestra(ruta):
    d = torch.load(ruta, map_location='cpu', weights_only=False)
    iq = d['x_iq'].float()
    target = int(ruta.split('_target')[1].split('_')[0])
    snr    = int(ruta.split('_snr')[1].replace('.pt',''))
    return iq, None, target, snr

def get_suelo_ms(h_seg, u_seg, dt):
    drop_max = np.max(u_seg - h_seg) + 1e-8
    mask = h_seg < (u_seg - 0.7 * drop_max)
    if not np.any(mask): return 0.0
    return max(len(list(g)) for k, g in itertools.groupby(mask) if k) * dt


In [ ]:
# ===== PARÁMETROS AJUSTABLES =====
TARGET_DESEADO = 3
SNR_DESEADA    = -14
TOL_DUR        = 0.2   # ±15% tolerancia de duración FHSS
# ==================================

DATA_DIR = r"C:\TFM_data\NoisyUAV\drone_RF_data"
archivos = glob.glob(os.path.join(DATA_DIR, f"*_target{TARGET_DESEADO}_snr{SNR_DESEADA}.pt"))
if not archivos:
    raise ValueError(f"❌ No hay ficheros target={TARGET_DESEADO} snr={SNR_DESEADA}")

semilla = random.randint(0, 9999)
# semilla = 2509   # fija para reproducibilidad
random.seed(semilla)
ruta = random.choice(archivos)
iq_tensor, _, original_target, original_snr = cargar_muestra(ruta)
print(f"🎬 Muestra: {os.path.basename(ruta)} (Semilla {semilla})")

# ── Detector de entropía ──────────────────────────────────────────────────────
# FS=14e6; NPERSEG=2048; Z_THRESH=2.0; MIN_BURST_MS=0.40; MERGE_GAP_MS=0.4
# MIN_Z_ABS=4.0; BG_MULT=4; MAX_BINS_FRAC=0.25; SMOOTH_MS=0.1; ADAPTIVE_WINDOW_MS=15

FS = 14e6
NPERSEG = 2048
Z_THRESH = 4.0      
MIN_BURST_MS = 0.5
MERGE_GAP_MS = 1.5
MIN_Z_ABS = 4.0
BG_MULT = 4
MAX_BINS_FRAC = 0.25
SMOOTH_MS = 0.2
ADAPTIVE_WINDOW_MS = 15 

t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS, min_z_abs=MIN_Z_ABS,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS)

# ── Duración mínima para este target ─────────────────────────────────────────
target_key = str(original_target)
dur_min_ref = 0.0
if target_key in DURATION_REF:
    dur_ref_mean = DURATION_REF[target_key]['dur_ref']
    dur_std      = DURATION_REF[target_key]['dur_std']
    print(f"📏 Referencia duración target {original_target}: "
          f"mean={dur_ref_mean:.2f}ms | std={dur_std:.2f}ms | "
          f"ventana=[{dur_ref_mean*(1-TOL_DUR):.2f} – {dur_ref_mean*(1+TOL_DUR):.2f}]ms")
else:
    print(f"ℹ️  Sin referencia de duración para target {original_target} — sin restricción mínima.")

print("=" * 70)
print("  ORÁCULO V9 — Plantilla FHSS + Restricción Duración Mínima")
print("=" * 70)

if len(bursts) == 0:
    print("  ✗ Sin transmisiones detectadas.")
else:
    global_nf      = float(np.median(nf_v))
    global_ns      = float(np.clip(ns, 0, 5))
    global_H_mean  = float(np.mean(H_smooth))
    global_p75_act = float(np.percentile(n_active, 75))
    dt = float(t_ms[1] - t_ms[0])

    # ── PASE 1: Inferencia IA en todos los bursts ─────────────────────────────
    with torch.no_grad():
        for b in bursts:
            h_seg = H_smooth[b['i0']:b['i1']+1]
            u_seg = umbral_v[b['i0']:b['i1']+1]
            b['suelo_ms']  = get_suelo_ms(h_seg, u_seg, dt)
            b['rugosidad'] = float(np.std(np.diff(h_seg))) if len(h_seg) > 1 else 0.5
            b['label_pseudo'] = -1
            b['rescate'] = None

            idx_i = int(b['t0']*1e-3*FS)
            idx_f = int(b['t1']*1e-3*FS)
            p_raw = iq_tensor[:, idx_i:idx_f]
            p_norm = p_raw / p_raw.pow(2).mean().clamp(min=1e-12).sqrt()
            T_LEN = int(9.4e-3*FS)
            p_pad = (torch.cat([p_norm, torch.zeros(2, T_LEN - p_norm.shape[1])], dim=1)
                     if p_norm.shape[1] < T_LEN else p_norm[:, :T_LEN])

            feat = torch.from_numpy(np.array([
                np.clip(b['dur_ms'], 0, 75),
                np.clip(abs(b['z_peak']), 0, 30),
                np.clip(b['drop_b'], 0, 10),
                np.clip(b['n_act'], 0, 2048),
                global_nf, global_ns, global_H_mean, global_p75_act
            ], dtype=np.float32)).to(DEVICE)
            feat_norm = torch.clamp((feat - phys_mean) / (phys_std + 1e-8), -5., 5.).unsqueeze(0)
            b['prob_ia'] = torch.sigmoid(
                model(p_pad.unsqueeze(0).to(DEVICE), feat_norm)
            ).item() * 100

    # ── PASE 2: Elegir CAMPEÓN ────────────────────────────────────────────────
    # Prioridad 1: El burst más parecido en duración a la referencia calibrada.
    # Prioridad 2: En caso de empate, el de mayor prob_ia.
    # Esto anula el error de elegir un BT click de 0.59ms como campeón
    # solo porque el Teacher (confundido por el ruido) le da 0.1% más.
    if target_key in DURATION_REF:
        campeon = min(
            bursts,
            key=lambda b: (
                abs(b['dur_ms'] - dur_ref_mean) / (dur_ref_mean + 1e-8),  # 1º: proximidad a referencia
                -b['prob_ia']                                              # 2º: mayor prob_ia (desempate)
            )
        )
        ref_dur = campeon['dur_ms']
        desv_camp = abs(campeon['dur_ms'] - dur_ref_mean) / (dur_ref_mean + 1e-8)
        if desv_camp > TOL_DUR:
            print(f"⚠️  Campeón a {desv_camp*100:.0f}% de la referencia "
                  f"(>{TOL_DUR*100:.0f}%). Considera subir TOL_DUR.")
    else:
        # Sin JSON de referencia: comportamiento V9 original (mayor prob_ia)
        campeon = max(bursts, key=lambda b: b['prob_ia'])
        print("ℹ️  Sin referencia de duración — campeón por mayor prob_ia.")

    # ── PASE 3: Clasificación por plantilla de duración FHSS ─────────────────
    for b in bursts:
        desv = abs(b['dur_ms'] - ref_dur) / (ref_dur + 1e-8)
        if b is campeon:
            b['label_pseudo'] = 1
            b['rescate'] = "🏆CAMPEÓN"
        elif desv <= TOL_DUR:
            b['label_pseudo'] = 1
            b['rescate'] = f"DUR≈ref ({desv*100:.1f}%off)"
        else:
            b['label_pseudo'] = 0
            b['rescate'] = f"DUR✗ {b['dur_ms']:.2f}≠{ref_dur:.2f}ms"

    # ── IMPRESIÓN ─────────────────────────────────────────────────────────────
    for i, b in enumerate(bursts):
        tag = "✅ DRON" if b['label_pseudo'] == 1 else "❌ RUIDO"
        print(f"  [B{i+1:02d}] t={b['t0']:6.2f}ms | dur={b['dur_ms']:.2f}ms | "
              f"suelo={b['suelo_ms']:.2f}ms | rug={b['rugosidad']:.3f} | "
              f"n_act={b['n_act']:.0f} | 🧠{b['prob_ia']:4.1f}% -> {tag} | {b['rescate']}")

print("=" * 70)


In [ ]:
fig = plot_muestra(iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
                   fs=FS, titulo=f"Oráculo V9 | target={original_target} | SNR={original_snr}dB")

for i, b in enumerate(bursts):
    c    = '#27ae60' if b.get('label_pseudo') == 1 else '#e74c3c'
    rgba = 'rgba(39,174,96,0.5)' if b.get('label_pseudo') == 1 else 'rgba(231,76,60,0.5)'
    for sh in fig.layout.shapes:
        if sh.type == 'rect' and sh.x0 == b['t0'] and sh.x1 == b['t1']:
            sh.fillcolor = c
    for an in fig.layout.annotations:
        if an.text == f"<b>B{i+1}</b>":
            an.font.color = c

fig.show()
